## AR / MA / ARMA / ARIMA / SARIMA 

This notebook runs AR($p$) (linear regression on lags) and statsmodels MA/ARMA/ARIMA/SARIMA models using walk-forward CV. All part can be run independently.

## AR($p$)

In [10]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from tqdm import tqdm
from sklearn.metrics import mean_squared_error
import os
from utils import extract, WFCV

def feature_engineering_arp(df, p):
    for lag in range(1, p + 1):
        df[f"lag_{lag}"] = df["log_return"].shift(lag)
    df = df.dropna()
    return df

class SimpleOLS:
    def __init__(self, add_intercept=True):
        self.add_intercept = add_intercept
        self.beta = None

    def fit(self, X, y):
        X_mat = X.values if isinstance(X, pd.DataFrame) else X
        if self.add_intercept:
            X_mat = np.column_stack([np.ones(len(X_mat)), X_mat])
        self.beta = np.linalg.pinv(X_mat.T @ X_mat) @ X_mat.T @ y

    def predict(self, X):
        X_mat = X.values if isinstance(X, pd.DataFrame) else X
        if self.add_intercept:
            X_mat = np.column_stack([np.ones(len(X_mat)), X_mat])
        return X_mat @ self.beta

def stats_forecasting_ar_custom(name_ticker, df_all, model, p, step_size, fold_size):
    df = extract(df_all, name_ticker) 

    df["log_return"] = np.log1p(df["return"])
    df["log_return"] = df["log_return"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["log_return"])

    df = feature_engineering_arp(df, p=p)
    features = [f"lag_{i}" for i in range(1, p + 1)]
    X = df[features]
    y = df["log_return"]

    y_pred, y_truth, mse_tab, r2 = WFCV(X, y, model, step_size=step_size, fold_size=fold_size)
    reg = sm.OLS(y_truth, sm.add_constant(y_pred)).fit()

    return {
        "SYMBOL": name_ticker,
        "MSE": float(np.mean(mse_tab)) if len(mse_tab) else np.nan,
        "OLS_R2": float(reg.rsquared),
        "OLS_Intercept": float(reg.params[0]),
        "OLS_Slope": float(reg.params[1]),
        "OLS_P_Value_Intercept": float(reg.pvalues[0]),
        "P_Value_Slope": float(reg.pvalues[1]),
    }

### 1 month window

In [11]:
if __name__ == "__main__":
    os.makedirs('Resultats', exist_ok=True)

    FOLD_SIZE = 21
    STEP_SIZE = 5
    MIN_REQUIS = FOLD_SIZE + STEP_SIZE + 20

    df_all = pd.read_csv('Datasets/returns_all.csv')
    df_all['Date'] = pd.to_datetime(df_all['Date'], errors='coerce')
    
    #tickers_to_drop = ['KOCRD Index', 'NOBRON Index', 'CABROVER Index', 'EB03/3M Index', 'BZSTSETA Index']
    #df_all.drop([t for t in tickers_to_drop if t in df_all.columns], axis=1, inplace=True, errors='ignore')
    
    all_tickers = df_all.columns[1:].tolist()

    valid_tickers = [col for col in all_tickers if df_all[col].count() >= MIN_REQUIS]
    print(f"Filtering completed: {len(valid_tickers)} tickers kept (those with >= {MIN_REQUIS} valid days).")

    model = SimpleOLS(add_intercept=True)
    ar_p_values = [1, 2, 5, 10]

    for p in tqdm(ar_p_values, desc='AR(p) Loop — 1 MONTH'):
        rows = []
        
        for name in tqdm(valid_tickers, desc=f'AR({p}) — 1 Month', leave=False):
            try:
                res = stats_forecasting_ar_custom(
                    name_ticker=name, 
                    df_all=df_all, 
                    model=model,
                    p=p,
                    step_size=STEP_SIZE, 
                    fold_size=FOLD_SIZE
                )
                rows.append(res)
            except ValueError as e:
                continue
            except Exception as e:
                print(f"\n[ERROR] {name} (AR {p}): {e}")
                continue
                
        df_p = pd.DataFrame(rows)
        csv_name = f"Resultats/resultats_all_tickers_AR({p})_f1month.csv"
        df_p.to_csv(csv_name, index=False)
        tqdm.write(f"Saved: {csv_name}")
        
    print("\nAll AR(p) calculations for 1 MONTH completed successfully!")

Filtering completed: 1066 tickers kept (those with >= 46 valid days).


AR(p) Loop — 1 MONTH:   0%|          | 0/4 [00:00<?, ?it/s]c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs,

Saved: Resultats/resultats_all_tickers_AR(1)_f1month.csv


c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-p

Saved: Resultats/resultats_all_tickers_AR(2)_f1month.csv


c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-p

Saved: Resultats/resultats_all_tickers_AR(5)_f1month.csv


c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-p

Saved: Resultats/resultats_all_tickers_AR(10)_f1month.csv

All AR(p) calculations for 1 MONTH completed successfully!


### 1 year window

In [13]:
if __name__ == "__main__":
    os.makedirs('Resultats', exist_ok=True)

    FOLD_SIZE = 200
    STEP_SIZE = 50
    MIN_REQUIS = FOLD_SIZE + STEP_SIZE + 20 

    df_all = pd.read_csv('Datasets/returns_all.csv')
    df_all['Date'] = pd.to_datetime(df_all['Date'], errors='coerce')
    
    tickers_to_drop = ['KOCRD Index', 'NOBRON Index', 'CABROVER Index', 'EB03/3M Index', 'BZSTSETA Index']
    df_all.drop([t for t in tickers_to_drop if t in df_all.columns], axis=1, inplace=True, errors='ignore')
    all_tickers = df_all.columns[1:].tolist()

    valid_tickers = [col for col in all_tickers if df_all[col].count() >= MIN_REQUIS]
    print(f"Filtering completed: {len(valid_tickers)} tickers kept (those with >= {MIN_REQUIS} valid days).")

    model = SimpleOLS(add_intercept=True)
    ar_p_values = [2, 5, 10]

    for p in tqdm(ar_p_values, desc='AR(p) Loop — 1 YEAR'):
        rows = []
        
        for name in tqdm(valid_tickers, desc=f'AR({p}) — 1 Year', leave=False):
            try:
                res = stats_forecasting_ar_custom(
                    name_ticker=name, 
                    df_all=df_all, 
                    model=model,
                    p=p,
                    step_size=STEP_SIZE, 
                    fold_size=FOLD_SIZE
                )
                rows.append(res)
            except ValueError as e:
                continue
            except Exception as e:
                print(f"\n[ERROR] {name} (AR {p}): {e}")
                continue
                
        df_p = pd.DataFrame(rows)
        csv_name = f"Resultats/resultats_all_tickers_AR({p})_f1year.csv"
        df_p.to_csv(csv_name, index=False)
        tqdm.write(f"Saved: {csv_name}")
        
    print("\nAll AR(p) calculations for 1 YEAR completed successfully!")

Filtering completed: 1061 tickers kept (those with >= 270 valid days).


AR(p) Loop — 1 YEAR:   0%|          | 0/3 [00:00<?, ?it/s]c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, 

Saved: Resultats/resultats_all_tickers_AR(2)_f1year.csv


c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-p

Saved: Resultats/resultats_all_tickers_AR(5)_f1year.csv


c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-p

Saved: Resultats/resultats_all_tickers_AR(10)_f1year.csv

All AR(p) calculations for 1 YEAR completed successfully!


### 5 years window

In [12]:
if __name__ == "__main__":
    os.makedirs('Resultats', exist_ok=True)

    FOLD_SIZE = 1260
    STEP_SIZE = 250
    MIN_REQUIS = FOLD_SIZE + STEP_SIZE + 20 

    df_all = pd.read_csv('Datasets/returns_all.csv')
    df_all['Date'] = pd.to_datetime(df_all['Date'], errors='coerce')
    
    #tickers_to_drop = ['KOCRD Index', 'NOBRON Index', 'CABROVER Index', 'EB03/3M Index', 'BZSTSETA Index']
    #df_all.drop([t for t in tickers_to_drop if t in df_all.columns], axis=1, inplace=True, errors='ignore')
    all_tickers = df_all.columns[1:].tolist()

    valid_tickers = [col for col in all_tickers if df_all[col].count() >= MIN_REQUIS]
    print(f"Filtering completed: {len(valid_tickers)} tickers kept (those with >= {MIN_REQUIS} valid days).")

    model = SimpleOLS(add_intercept=True)
    ar_p_values = [2, 5, 10]

    for p in tqdm(ar_p_values, desc='AR(p) Loop — 5 YEARS'):
        rows = []
        
        for name in tqdm(valid_tickers, desc=f'AR({p}) — 5 Years', leave=False):
            try:
                res = stats_forecasting_ar_custom(
                    name_ticker=name, 
                    df_all=df_all, 
                    model=model,
                    p=p,
                    step_size=STEP_SIZE, 
                    fold_size=FOLD_SIZE
                )
                rows.append(res)
            except ValueError as e:
                continue
            except Exception as e:
                print(f"\n[ERROR] {name} (AR {p}): {e}")
                continue
                
        df_p = pd.DataFrame(rows)
        csv_name = f"Resultats/resultats_all_tickers_AR({p})_f5years.csv"
        df_p.to_csv(csv_name, index=False)
        tqdm.write(f"Saved: {csv_name}")
        
    print("\nAll AR(p) calculations for 5 YEARS completed successfully!")

Filtering completed: 1013 tickers kept (those with >= 1530 valid days).


AR(p) Loop — 5 YEARS:   0%|          | 0/3 [00:00<?, ?it/s]c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs,

Saved: Resultats/resultats_all_tickers_AR(2)_f5years.csv


c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-p

Saved: Resultats/resultats_all_tickers_AR(5)_f5years.csv


c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-p

Saved: Resultats/resultats_all_tickers_AR(10)_f5years.csv

All AR(p) calculations for 5 YEARS completed successfully!


## MA($q$)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.optimize import minimize
from scipy.signal import lfilter
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
import warnings


class SimpleMA:
    def __init__(self, q=1):
        self.q = int(q)
        self.mu = 0.0
        self.thetas = np.zeros(self.q)
        self.last_resids = np.zeros(self.q)

    def _objective(self, params, y):
        mu = params[0]
        a = np.insert(params[1:], 0, 1.0)
        resid = lfilter([1.0], a, y - mu)
        
        sse = np.sum(resid**2)

        if not np.isfinite(sse) or sse > 1e10:
            return 1e10
            
        return sse

    def fit(self, y):
        initial_guess = np.zeros(self.q + 1)
        initial_guess[0] = np.mean(y)
        
        bounds = [(None, None)] + [(-0.99, 0.99)] * self.q
        
        res = minimize(
            self._objective, 
            initial_guess, 
            args=(y,), 
            method='L-BFGS-B',
            bounds=bounds,
            options={'maxiter': 50, 'ftol': 1e-4}
        )
        
        self.mu = res.x[0]
        self.thetas = res.x[1:]
        
        a = np.insert(self.thetas, 0, 1.0)
        final_resids = lfilter([1.0], a, y - self.mu)
        
        if len(final_resids) >= self.q:
            self.last_resids = final_resids[-self.q:]
        else:
            pad = np.zeros(self.q - len(final_resids))
            self.last_resids = np.concatenate([pad, final_resids])
            
        return self

    def predict(self, steps=1):
        preds = np.zeros(steps)
        e_array = np.concatenate([self.last_resids, np.zeros(steps)])
        
        for h in range(steps):
            sum_ma = 0
            for i in range(1, self.q + 1):
                sum_ma += self.thetas[i-1] * e_array[self.q + h - i]
            
            preds[h] = self.mu + sum_ma
            
        return preds


def wfcv_ma(y, q=1, step_size=50, fold_size=200):
    y_vals = y.values
    n = len(y_vals)
    
    preds = []
    truths = []
    mse_tab = []
    
    model = SimpleMA(q=q)
    
    for start in range(0, n - fold_size - step_size + 1, step_size):
        y_train = y_vals[start : start + fold_size]
        y_test = y_vals[start + fold_size : start + fold_size + step_size]
        
        try:
            model.fit(y_train)
            fcst = model.predict(steps=len(y_test))
            mse_tab.append(mean_squared_error(y_test, fcst))
        except Exception:
            fcst = [np.nan] * len(y_test)
            mse_tab.append(np.nan)
            
        preds.extend(fcst)
        truths.extend(y_test)
        
    return np.array(preds), np.array(truths), np.array(mse_tab)


def extract(df_all, ticker):
    df_t = df_all[['Date', ticker]].copy()
    df_t = df_t.rename(columns={ticker: 'return'})
    return df_t

def stats_forecasting_ma_custom(name_ticker, df_all, q=1, step_size=50, fold_size=200):
    df_t = extract(df_all, name_ticker)
    
    df_t = df_t[df_t['return'] > -1].copy()
    df_t['log_return'] = np.log1p(df_t['return'])
    df_t['log_return'] = df_t['log_return'].replace([np.inf, -np.inf], np.nan)
    df_t = df_t.dropna(subset=['log_return'])
    
    y = df_t['log_return']
    
    y_pred, y_true, mse_tab = wfcv_ma(y=y, q=q, step_size=step_size, fold_size=fold_size)

    mask = np.isfinite(y_pred) & np.isfinite(y_true)
    if mask.sum() >= 3:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            reg = sm.OLS(y_true[mask], sm.add_constant(y_pred[mask])).fit()
            ols_r2 = float(reg.rsquared)
            ols_intercept = float(reg.params[0])
            ols_slope = float(reg.params[1])
            p_int = float(reg.pvalues[0])
            p_slope = float(reg.pvalues[1])
    else:
        ols_r2 = ols_intercept = ols_slope = p_int = p_slope = float('nan')

    return {
        'SYMBOL': name_ticker,
        'Model': f'MA({q}) Custom',
        'q': q,
        'MSE': float(np.nanmean(mse_tab)) if len(mse_tab) > 0 else np.nan,
        'OLS_R2': ols_r2,
        'OLS_Intercept': ols_intercept,
        'OLS_Slope': ols_slope,
        'OLS_P_Value_Intercept': p_int,
        'P_Value_Slope': p_slope,
        'n_obs': int(len(y)),
    }


### Window 1 year

In [ ]:
if __name__ == "__main__":
    STEP_SIZE = 50
    FOLD_SIZE = 200

    df_all = pd.read_csv('Datasets/returns_all.csv')
    df_all['Date'] = pd.to_datetime(df_all['Date'], errors='coerce')
    
    all_tickers = df_all.columns[1:].tolist()

    ma_qs = [2, 5, 10]
    results_ma = {}

    for q in tqdm(ma_qs, desc='Boucle MA(q) — différents q'):
        rows = []
        
        for name in tqdm(all_tickers, desc=f'MA({q}) — all tickers', leave=False):
            res = stats_forecasting_ma_custom(
                name_ticker=name, 
                df_all=df_all, 
                q=q,
                step_size=STEP_SIZE, 
                fold_size=FOLD_SIZE
            )
            rows.append(res)
            
        df_q = pd.DataFrame(rows)
        results_ma[q] = df_q
        df_q.to_csv(f"Resultats/resultats_all_tickers_MA({q}).csv", index=False)
        
    print("\nEnd for all MA(q).")

### Window 1 month

In [4]:
if __name__ == "__main__":
    import os
    os.makedirs('Resultats', exist_ok=True)

    FOLD_SIZE = 21
    STEP_SIZE = 5
    MIN_REQUIS = FOLD_SIZE + STEP_SIZE + 5 

    df_all = pd.read_csv('Datasets/returns_all.csv')
    df_all['Date'] = pd.to_datetime(df_all['Date'], errors='coerce')
    all_tickers = df_all.columns[1:].tolist()

    valid_tickers = [col for col in all_tickers if df_all[col].count() >= MIN_REQUIS]
    print(f"{len(valid_tickers)} tickers kept (those with >= {MIN_REQUIS} valid days).")

    ma_qs = [1, 2, 5, 10]
    results_ma_1m = {}

    for q in tqdm(ma_qs, desc='Loop MA(q) — window 1 month'):
        rows = []
        
        for name in tqdm(valid_tickers, desc=f'MA({q}) — 1 month', leave=False):
            try:
                res = stats_forecasting_ma_custom(
                    name_ticker=name, 
                    df_all=df_all, 
                    q=q,               
                    step_size=STEP_SIZE, 
                    fold_size=FOLD_SIZE
                )
                rows.append(res)
            except ValueError as e:
                continue
            except Exception as e:
                print(f"\n[ERROR] {name} (MA {q}): {e}")
                continue
                
        df_q = pd.DataFrame(rows)
        results_ma_1m[q] = df_q
        csv_name = f"Resultats/resultats_all_tickers_MA({q})_f1month.csv"
        df_q.to_csv(csv_name, index=False)
        tqdm.write(f"Saved : {csv_name}")
        
    print("\nEnd with success for all MA(q) on a 1 month window")

1066 tickers kept (those with >= 31 valid days).


Loop MA(q) — window 1 month: 100%|██████████| 1/1 [15:08<00:00, 908.70s/it]

Saved : Resultats/resultats_all_tickers_MA(1)_f1month.csv

End with success for all MA(q) on a 1 month window


### Window 5 years

In [5]:
if __name__ == "__main__":
    import os
    os.makedirs('Resultats', exist_ok=True)

    FOLD_SIZE = 1260
    STEP_SIZE = 250
    MIN_REQUIS = FOLD_SIZE + STEP_SIZE + 100

    df_all = pd.read_csv('Datasets/returns_all.csv')
    df_all['Date'] = pd.to_datetime(df_all['Date'], errors='coerce')
    all_tickers = df_all.columns[1:].tolist()

    valid_tickers = [col for col in all_tickers if df_all[col].count() >= MIN_REQUIS]
    print(f"{len(valid_tickers)} tickers kept (those with >= {MIN_REQUIS} valid days).")

    ma_qs = [1, 2, 5, 10]
    results_ma_1m = {}

    for q in tqdm(ma_qs, desc='Loop MA(q) — window 5 years'):
        rows = []
        
        for name in tqdm(valid_tickers, desc=f'MA({q}) — 5years', leave=False):
            try:
                res = stats_forecasting_ma_custom(
                    name_ticker=name, 
                    df_all=df_all, 
                    q=q,               
                    step_size=STEP_SIZE, 
                    fold_size=FOLD_SIZE
                )
                rows.append(res)
            except ValueError as e:
                continue
            except Exception as e:
                print(f"\n[ERROR] {name} (MA {q}): {e}")
                continue
                
        df_q = pd.DataFrame(rows)
        results_ma_1m[q] = df_q
        csv_name = f"Resultats/resultats_all_tickers_MA({q})_f5years.csv"
        df_q.to_csv(csv_name, index=False)
        tqdm.write(f"Saved : {csv_name}")
        
    print("\nEnd with success for all MA(q) on a 5 years window")

1012 tickers kept (those with >= 1610 valid days).


Loop MA(q) — window 5 years: 100%|██████████| 1/1 [00:22<00:00, 22.36s/it]

Saved : Resultats/resultats_all_tickers_MA(1)_f5years.csv

End with success for all MA(q) on a 5 years window


## ARMA($p, q$)

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.optimize import minimize
from scipy.signal import lfilter
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
import warnings


class SimpleARMA:
    def __init__(self, p=1, q=1):
        self.p = int(p)
        self.q = int(q)
        self.mu = 0.0
        self.phis = np.zeros(self.p)    # Coefficients AR
        self.thetas = np.zeros(self.q)  # Coefficients MA
        self.last_y_centered = np.zeros(self.p) # Historique des Y (centrés)
        self.last_resids = np.zeros(self.q)     # Historique des erreurs

    def _objective(self, params, y):
        mu = params[0]
        phis = params[1 : 1 + self.p]
        thetas = params[1 + self.p :]
        
        # Filtre lfilter(b, a, x) :
        # b (numérateur)   = [1, -phi_1, -phi_2, ...]   => Partie AR
        # a (dénominateur) = [1, theta_1, theta_2, ...] => Partie MA
        b = np.insert(-phis, 0, 1.0)
        a = np.insert(thetas, 0, 1.0)
        
        resid = lfilter(b, a, y - mu)
        
        sse = np.sum(resid**2)
        
        if not np.isfinite(sse) or sse > 1e10:
            return 1e10
            
        return sse

    def fit(self, y):
        initial_guess = np.zeros(1 + self.p + self.q)
        initial_guess[0] = np.mean(y)
        
        bounds = [(None, None)] + [(-0.99, 0.99)] * (self.p + self.q)
        
        res = minimize(
            self._objective, 
            initial_guess, 
            args=(y,), 
            method='L-BFGS-B',
            bounds=bounds,
            options={'maxiter': 50, 'ftol': 1e-4}
        )
        
        self.mu = res.x[0]
        self.phis = res.x[1 : 1 + self.p]
        self.thetas = res.x[1 + self.p :]
        
        y_centered = y - self.mu
        b = np.insert(-self.phis, 0, 1.0)
        a = np.insert(self.thetas, 0, 1.0)
        final_resids = lfilter(b, a, y_centered)
        
        if self.p > 0:
            if len(y_centered) >= self.p:
                self.last_y_centered = y_centered[-self.p:]
            else:
                self.last_y_centered = np.concatenate([np.zeros(self.p - len(y_centered)), y_centered])
                
        if self.q > 0:
            if len(final_resids) >= self.q:
                self.last_resids = final_resids[-self.q:]
            else:
                self.last_resids = np.concatenate([np.zeros(self.q - len(final_resids)), final_resids])
            
        return self

    def predict(self, steps=1):
        preds = np.zeros(steps)
        
        y_hist = np.concatenate([self.last_y_centered, np.zeros(steps)])
        e_hist = np.concatenate([self.last_resids, np.zeros(steps)])
        
        for h in range(steps):
            ar_sum = 0
            for i in range(1, self.p + 1):
                ar_sum += self.phis[i-1] * y_hist[self.p + h - i]
                
            ma_sum = 0
            for j in range(1, self.q + 1):
                ma_sum += self.thetas[j-1] * e_hist[self.q + h - j]
            
            pred_centered = ar_sum + ma_sum
            
            y_hist[self.p + h] = pred_centered
            
            preds[h] = self.mu + pred_centered
            
        return preds


def wfcv_arma(y, p=1, q=1, step_size=50, fold_size=200):
    y_vals = y.values
    n = len(y_vals)
    
    preds = []
    truths = []
    mse_tab = []
    
    model = SimpleARMA(p=p, q=q)
    
    for start in range(0, n - fold_size - step_size + 1, step_size):
        y_train = y_vals[start : start + fold_size]
        y_test = y_vals[start + fold_size : start + fold_size + step_size]
        
        try:
            model.fit(y_train)
            fcst = model.predict(steps=len(y_test))
            mse_tab.append(mean_squared_error(y_test, fcst))
        except Exception:
            fcst = [np.nan] * len(y_test)
            mse_tab.append(np.nan)
            
        preds.extend(fcst)
        truths.extend(y_test)
        
    return np.array(preds), np.array(truths), np.array(mse_tab)


def extract(df_all, ticker):
    df_t = df_all[['Date', ticker]].copy()
    df_t = df_t.rename(columns={ticker: 'return'})
    return df_t

def stats_forecasting_arma_custom(name_ticker, df_all, p=1, q=1, step_size=50, fold_size=200):
    df_t = extract(df_all, name_ticker)
    
    df_t = df_t[df_t['return'] > -1].copy()
    df_t['log_return'] = np.log1p(df_t['return'])
    df_t['log_return'] = df_t['log_return'].replace([np.inf, -np.inf], np.nan)
    df_t = df_t.dropna(subset=['log_return'])
    
    y = df_t['log_return']
    
    y_pred, y_true, mse_tab = wfcv_arma(y=y, p=p, q=q, step_size=step_size, fold_size=fold_size)

    mask = np.isfinite(y_pred) & np.isfinite(y_true)
    if mask.sum() >= 3:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            reg = sm.OLS(y_true[mask], sm.add_constant(y_pred[mask])).fit()
            ols_r2 = float(reg.rsquared)
            ols_intercept = float(reg.params[0])
            ols_slope = float(reg.params[1])
            p_int = float(reg.pvalues[0])
            p_slope = float(reg.pvalues[1])
    else:
        ols_r2 = ols_intercept = ols_slope = p_int = p_slope = float('nan')

    return {
        'SYMBOL': name_ticker,
        'Model': f'ARMA({p},{q})',
        'p': p,
        'q': q,
        'MSE': float(np.nanmean(mse_tab)) if len(mse_tab) > 0 else np.nan,
        'OLS_R2': ols_r2,
        'OLS_Intercept': ols_intercept,
        'OLS_Slope': ols_slope,
        'OLS_P_Value_Intercept': p_int,
        'P_Value_Slope': p_slope,
        'n_obs': int(len(y)),
    }

### Window 1 year

In [ ]:
if __name__ == "__main__":
    STEP_SIZE = 50
    FOLD_SIZE = 200

    df_all = pd.read_csv('Datasets/returns_all.csv')
    df_all['Date'] = pd.to_datetime(df_all['Date'], errors='coerce')
    
    all_tickers = df_all.columns[1:].tolist()

    arma_orders = [(1, 1), (2, 2), (5, 5)]
    results_arma = {}

    for p, q in tqdm(arma_orders, desc='Loop ARMA(p,q)'):
        rows = []
        
        for name in tqdm(all_tickers, desc=f'ARMA({p},{q}) — all tickers', leave=False):
            res = stats_forecasting_arma_custom(
                name_ticker=name, 
                df_all=df_all, 
                p=p, 
                q=q,
                step_size=STEP_SIZE, 
                fold_size=FOLD_SIZE
            )
            rows.append(res)
            
        df_pq = pd.DataFrame(rows)
        results_arma[(p, q)] = df_pq
        df_pq.to_csv(f"Resultats/resultats_all_tickers_ARMA({p},{q}).csv", index=False)
        
    print("\nARMA(p,q) computations are finished for window 1year!")

### Window 1 month

In [2]:
if __name__ == "__main__":
    STEP_SIZE = 5
    FOLD_SIZE = 21

    df_all = pd.read_csv('Datasets/returns_all.csv')
    df_all['Date'] = pd.to_datetime(df_all['Date'], errors='coerce')
    
    all_tickers = df_all.columns[1:].tolist()

    arma_orders = [(1, 1), (2, 2), (5, 5)]
    results_arma = {}

    for p, q in tqdm(arma_orders, desc='Loop ARMA(p,q)'):
        rows = []
        
        for name in tqdm(all_tickers, desc=f'ARMA({p},{q}) — all tickers - window 1 month', leave=False):
            res = stats_forecasting_arma_custom(
                name_ticker=name, 
                df_all=df_all, 
                p=p, 
                q=q,
                step_size=STEP_SIZE, 
                fold_size=FOLD_SIZE
            )
            rows.append(res)
            
        df_pq = pd.DataFrame(rows)
        results_arma[(p, q)] = df_pq
        df_pq.to_csv(f"Resultats/resultats_all_tickers_ARMA({p},{q})_f1month.csv", index=False)
        
    print("\nARMA(p,q) computations are finished for window 1 month!")

Loop ARMA(p,q): 100%|██████████| 3/3 [2:47:43<00:00, 3354.44s/it]


ARMA(p,q) computations are finished for window 1 month!


### Window 5 years

In [3]:
if __name__ == "__main__":
    STEP_SIZE = 250
    FOLD_SIZE = 1260

    df_all = pd.read_csv('Datasets/returns_all.csv')
    df_all['Date'] = pd.to_datetime(df_all['Date'], errors='coerce')
    
    all_tickers = df_all.columns[1:].tolist()

    arma_orders = [(1, 1), (2, 2), (5, 5)]
    results_arma = {}

    for p, q in tqdm(arma_orders, desc='Loop ARMA(p,q) - window 5 years'):
        rows = []
        
        for name in tqdm(all_tickers, desc=f'ARMA({p},{q}) — all tickers', leave=False):
            res = stats_forecasting_arma_custom(
                name_ticker=name, 
                df_all=df_all, 
                p=p, 
                q=q,
                step_size=STEP_SIZE, 
                fold_size=FOLD_SIZE
            )
            rows.append(res)
            
        df_pq = pd.DataFrame(rows)
        results_arma[(p, q)] = df_pq
        df_pq.to_csv(f"Resultats/resultats_all_tickers_ARMA({p},{q})_f5years.csv", index=False)
        
    print("\nARMA(p,q) computations are finished for window 5 years!")

Loop ARMA(p,q) - window 5 years:  33%|███▎      | 1/3 [00:25<00:50, 25.04s/it]C:\Users\diane\AppData\Local\Temp\ipykernel_18664\123236224.py:34: RuntimeWarning: overflow encountered in square
  sse = np.sum(resid**2)
C:\Users\diane\AppData\Local\Temp\ipykernel_18664\123236224.py:34: RuntimeWarning: overflow encountered in square
  sse = np.sum(resid**2)
C:\Users\diane\AppData\Local\Temp\ipykernel_18664\123236224.py:34: RuntimeWarning: overflow encountered in square
  sse = np.sum(resid**2)
C:\Users\diane\AppData\Local\Temp\ipykernel_18664\123236224.py:34: RuntimeWarning: overflow encountered in square
  sse = np.sum(resid**2)
C:\Users\diane\AppData\Local\Temp\ipykernel_18664\123236224.py:34: RuntimeWarning: overflow encountered in square
  sse = np.sum(resid**2)
C:\Users\diane\AppData\Local\Temp\ipykernel_18664\123236224.py:34: RuntimeWarning: overflow encountered in square
  sse = np.sum(resid**2)
C:\Users\diane\AppData\Local\Temp\ipykernel_18664\123236224.py:34: RuntimeWarning: overf


ARMA(p,q) computations are finished for window 5 years!


## ARIMA($p, d, q$)

In [4]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.optimize import minimize
from scipy.signal import lfilter
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
import warnings
import os

class SimpleARMA: # we replicate the ARMA class used below because we need it for the ARIMA class
    def __init__(self, p=1, q=1):
        self.p = int(p)
        self.q = int(q)
        self.mu = 0.0
        self.phis = np.zeros(self.p)
        self.thetas = np.zeros(self.q)
        self.last_y_centered = np.zeros(self.p)
        self.last_resids = np.zeros(self.q)

    def _objective(self, params, y):
        mu = params[0]
        phis = params[1 : 1 + self.p]
        thetas = params[1 + self.p :]
        
        b = np.insert(-phis, 0, 1.0)
        a = np.insert(thetas, 0, 1.0)
        
        resid = lfilter(b, a, y - mu)
        sse = np.sum(resid**2)
        
        if not np.isfinite(sse) or sse > 1e10:
            return 1e10
            
        return sse

    def fit(self, y):
        initial_guess = np.zeros(1 + self.p + self.q)
        initial_guess[0] = np.mean(y)
        bounds = [(None, None)] + [(-0.99, 0.99)] * (self.p + self.q)
        
        res = minimize(
            self._objective, 
            initial_guess, 
            args=(y,), 
            method='L-BFGS-B',
            bounds=bounds,
            options={'maxiter': 50, 'ftol': 1e-4}
        )
        
        self.mu = res.x[0]
        self.phis = res.x[1 : 1 + self.p]
        self.thetas = res.x[1 + self.p :]
        
        y_centered = y - self.mu
        b = np.insert(-self.phis, 0, 1.0)
        a = np.insert(self.thetas, 0, 1.0)
        final_resids = lfilter(b, a, y_centered)
        
        if self.p > 0:
            if len(y_centered) >= self.p:
                self.last_y_centered = y_centered[-self.p:]
            else:
                self.last_y_centered = np.concatenate([np.zeros(self.p - len(y_centered)), y_centered])
                
        if self.q > 0:
            if len(final_resids) >= self.q:
                self.last_resids = final_resids[-self.q:]
            else:
                self.last_resids = np.concatenate([np.zeros(self.q - len(final_resids)), final_resids])
            
        return self

    def predict(self, steps=1):
        preds = np.zeros(steps)
        y_hist = np.concatenate([self.last_y_centered, np.zeros(steps)])
        e_hist = np.concatenate([self.last_resids, np.zeros(steps)])
        
        for h in range(steps):
            ar_sum = 0
            for i in range(1, self.p + 1):
                ar_sum += self.phis[i-1] * y_hist[self.p + h - i]
                
            ma_sum = 0
            for j in range(1, self.q + 1):
                ma_sum += self.thetas[j-1] * e_hist[self.q + h - j]
            
            pred_centered = ar_sum + ma_sum
            y_hist[self.p + h] = pred_centered
            preds[h] = self.mu + pred_centered
            
        return preds


class SimpleARIMA:
    def __init__(self, p=1, d=0, q=1):
        self.p = int(p)
        self.d = int(d)
        self.q = int(q)
        self.arma_model = SimpleARMA(p=self.p, q=self.q)
        self.last_y_history = np.array([]) 

    def fit(self, y):
        if len(y) <= self.d:
            raise ValueError(f"Pas assez de données pour différencier {self.d} fois.")
            
        if self.d > 0:
            self.last_y_history = y[-self.d:].copy()
            
        y_diff = y.copy()
        for _ in range(self.d):
            y_diff = np.diff(y_diff)
            
        self.arma_model.fit(y_diff)
        return self

    def predict(self, steps=1):
        diff_preds = self.arma_model.predict(steps=steps)
        
        if self.d == 0:
            return diff_preds
            
        last_vals = []
        current_series = self.last_y_history.copy()
        for _ in range(self.d):
            last_vals.append(current_series[-1])
            if len(current_series) > 1:
                current_series = np.diff(current_series)
                
        preds = np.array(diff_preds)
        for i in range(self.d - 1, -1, -1):
            preds = last_vals[i] + np.cumsum(preds)
            
        return preds


def wfcv_arima(y, p=1, d=0, q=1, step_size=50, fold_size=200):
    y_vals = y.values
    n = len(y_vals)
    
    preds = []
    truths = []
    mse_tab = []
    
    model = SimpleARIMA(p=p, d=d, q=q)
    
    for start in range(0, n - fold_size - step_size + 1, step_size):
        y_train = y_vals[start : start + fold_size]
        y_test = y_vals[start + fold_size : start + fold_size + step_size]
        
        try:
            model.fit(y_train)
            fcst = model.predict(steps=len(y_test))
            mse_tab.append(mean_squared_error(y_test, fcst))
        except Exception:
            fcst = [np.nan] * len(y_test)
            mse_tab.append(np.nan)
            
        preds.extend(fcst)
        truths.extend(y_test)
        
    return np.array(preds), np.array(truths), np.array(mse_tab)

def extract(df_all, ticker):
    df_t = df_all[['Date', ticker]].copy()
    df_t = df_t.rename(columns={ticker: 'return'})
    return df_t

def stats_forecasting_arima_custom(name_ticker, df_all, p=1, d=0, q=1, step_size=50, fold_size=200):
    df_t = extract(df_all, name_ticker)
    
    df_t = df_t[df_t['return'] > -1].copy()
    df_t['log_return'] = np.log1p(df_t['return'])
    df_t['log_return'] = df_t['log_return'].replace([np.inf, -np.inf], np.nan)
    df_t = df_t.dropna(subset=['log_return'])
    
    y = df_t['log_return']
    
    y_pred, y_true, mse_tab = wfcv_arima(y=y, p=p, d=d, q=q, step_size=step_size, fold_size=fold_size)

    mask = np.isfinite(y_pred) & np.isfinite(y_true)
    if mask.sum() >= 3:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            reg = sm.OLS(y_true[mask], sm.add_constant(y_pred[mask])).fit()
            ols_r2 = float(reg.rsquared)
            ols_intercept = float(reg.params[0])
            ols_slope = float(reg.params[1])
            p_int = float(reg.pvalues[0])
            p_slope = float(reg.pvalues[1])
    else:
        ols_r2 = ols_intercept = ols_slope = p_int = p_slope = float('nan')

    return {
        'SYMBOL': name_ticker,
        'Model': f'ARIMA({p},{d},{q}) Custom',
        'p': p,
        'd': d,
        'q': q,
        'MSE': float(np.nanmean(mse_tab)) if len(mse_tab) > 0 else np.nan,
        'OLS_R2': ols_r2,
        'OLS_Intercept': ols_intercept,
        'OLS_Slope': ols_slope,
        'OLS_P_Value_Intercept': p_int,
        'P_Value_Slope': p_slope,
        'n_obs': int(len(y)),
    }


### Window 1 month

In [5]:
if __name__ == "__main__":
    import os
    os.makedirs('Resultats', exist_ok=True)

    FOLD_SIZE = 21 
    STEP_SIZE = 5
    MIN_REQUIS = FOLD_SIZE + STEP_SIZE + 5 

    df_all = pd.read_csv('Datasets/returns_all.csv')
    df_all['Date'] = pd.to_datetime(df_all['Date'], errors='coerce')
    all_tickers = df_all.columns[1:].tolist()

    valid_tickers = [col for col in all_tickers if df_all[col].count() >= MIN_REQUIS]
    print(f"Filtering completed: {len(valid_tickers)} tickers kept (those with >= {MIN_REQUIS} valid days).")

    arima_orders = [(1, 1, 1), (2, 1, 2)]
    results_arima_1m = {}

    for p, d, q in tqdm(arima_orders, desc='ARIMA(p,d,q) Loop — 1 MONTH'):
        rows = []
        
        for name in tqdm(valid_tickers, desc=f'ARIMA({p},{d},{q}) — 1 Month', leave=False):
            try:
                res = stats_forecasting_arima_custom(
                    name_ticker=name, 
                    df_all=df_all, 
                    p=p, d=d, q=q,
                    step_size=STEP_SIZE, 
                    fold_size=FOLD_SIZE
                )
                rows.append(res)
            except ValueError as e:
                continue
            except Exception as e:
                print(f"\n[ERROR] {name} (ARIMA {p},{d},{q}): {e}")
                continue
                
        df_order = pd.DataFrame(rows)
        results_arima_1m[(p, d, q)] = df_order
        csv_name = f"Resultats/resultats_all_tickers_ARIMA({p},{d},{q})_f1month.csv"
        df_order.to_csv(csv_name, index=False)
        tqdm.write(f"Saved: {csv_name}")
        
    print("\nAll ARIMA calculations for 1 month completed successfully!")

Filtering completed: 1066 tickers kept (those with >= 31 valid days).


ARIMA(p,d,q) Loop — 1 MONTH:  50%|█████     | 1/2 [18:13<18:13, 1093.85s/it]

Saved: Resultats/resultats_all_tickers_ARIMA(1,1,1)_f1month.csv


ARIMA(p,d,q) Loop — 1 MONTH: 100%|██████████| 2/2 [44:29<00:00, 1334.50s/it]

Saved: Resultats/resultats_all_tickers_ARIMA(2,1,2)_f1month.csv

All ARIMA calculations for 1 month completed successfully!


### Window 1 year

In [ ]:
if __name__ == "__main__":
    import os
    os.makedirs('Resultats', exist_ok=True)

    FOLD_SIZE = 200  
    STEP_SIZE = 50
    MIN_REQUIS = FOLD_SIZE + STEP_SIZE + 20 

    df_all = pd.read_csv('Datasets/returns_all.csv')
    df_all['Date'] = pd.to_datetime(df_all['Date'], errors='coerce')
    all_tickers = df_all.columns[1:].tolist()

    valid_tickers = [col for col in all_tickers if df_all[col].count() >= MIN_REQUIS]
    print(f"Filtering completed: {len(valid_tickers)} tickers kept (those with >= {MIN_REQUIS} valid days).")

    arima_orders = [(1, 1, 1), (2, 1, 2)]
    results_arima_1y = {}

    for p, d, q in tqdm(arima_orders, desc='ARIMA(p,d,q) Loop — 1 YEAR'):
        rows = []
        
        for name in tqdm(valid_tickers, desc=f'ARIMA({p},{d},{q}) — 1 Year', leave=False):
            try:
                res = stats_forecasting_arima_custom(
                    name_ticker=name, 
                    df_all=df_all, 
                    p=p, d=d, q=q,
                    step_size=STEP_SIZE, 
                    fold_size=FOLD_SIZE
                )
                rows.append(res)
            except ValueError as e:
                continue
            except Exception as e:
                print(f"\n[ERROR] {name} (ARIMA {p},{d},{q}): {e}")
                continue
                
        # Save results for this specific parameter combination
        df_order = pd.DataFrame(rows)
        results_arima_1y[(p, d, q)] = df_order
        csv_name = f"Resultats/resultats_all_tickers_ARIMA({p},{d},{q})_f1year.csv"
        df_order.to_csv(csv_name, index=False)
        tqdm.write(f"Saved: {csv_name}")
        
    print("\nAll ARIMA calculations for 1 YEAR completed successfully!")

### Window 5 years

In [6]:
if __name__ == "__main__":
    import os
    os.makedirs('Resultats', exist_ok=True)

    FOLD_SIZE = 1260
    STEP_SIZE = 250
    MIN_REQUIS = FOLD_SIZE + STEP_SIZE + 100 

    df_all = pd.read_csv('Datasets/returns_all.csv')
    df_all['Date'] = pd.to_datetime(df_all['Date'], errors='coerce')
    all_tickers = df_all.columns[1:].tolist()

    valid_tickers = [col for col in all_tickers if df_all[col].count() >= MIN_REQUIS]
    print(f"Filtering completed: {len(valid_tickers)} tickers kept (those with >= {MIN_REQUIS} valid days).")

    arima_orders = [(1, 1, 1), (2, 1, 2)]
    results_arima_5y = {}

    for p, d, q in tqdm(arima_orders, desc='ARIMA(p,d,q) Loop — 5 YEARS'):
        rows = []
        
        for name in tqdm(valid_tickers, desc=f'ARIMA({p},{d},{q}) — 5 Years', leave=False):
            try:
                res = stats_forecasting_arima_custom(
                    name_ticker=name, 
                    df_all=df_all, 
                    p=p, d=d, q=q,
                    step_size=STEP_SIZE, 
                    fold_size=FOLD_SIZE
                )
                rows.append(res)
            except ValueError as e:
                continue
            except Exception as e:
                print(f"\n[ERROR] {name} (ARIMA {p},{d},{q}): {e}")
                continue
                
        # Save results for this specific parameter combination
        df_order = pd.DataFrame(rows)
        results_arima_5y[(p, d, q)] = df_order
        csv_name = f"Resultats/resultats_all_tickers_ARIMA({p},{d},{q})_f5years.csv"
        df_order.to_csv(csv_name, index=False)
        tqdm.write(f"Saved: {csv_name}")
        
    print("\nAll ARIMA calculations for 5 YEARS completed successfully!")

Filtering completed: 1012 tickers kept (those with >= 1610 valid days).


ARIMA(p,d,q) Loop — 5 YEARS:  50%|█████     | 1/2 [00:30<00:30, 30.54s/it]

Saved: Resultats/resultats_all_tickers_ARIMA(1,1,1)_f5years.csv


C:\Users\diane\AppData\Local\Temp\ipykernel_18664\1489119682.py:30: RuntimeWarning: overflow encountered in square
  sse = np.sum(resid**2)
C:\Users\diane\AppData\Local\Temp\ipykernel_18664\1489119682.py:30: RuntimeWarning: overflow encountered in square
  sse = np.sum(resid**2)
C:\Users\diane\AppData\Local\Temp\ipykernel_18664\1489119682.py:30: RuntimeWarning: overflow encountered in square
  sse = np.sum(resid**2)
C:\Users\diane\AppData\Local\Temp\ipykernel_18664\1489119682.py:30: RuntimeWarning: overflow encountered in square
  sse = np.sum(resid**2)
C:\Users\diane\AppData\Local\Temp\ipykernel_18664\1489119682.py:30: RuntimeWarning: overflow encountered in square
  sse = np.sum(resid**2)
C:\Users\diane\AppData\Local\Temp\ipykernel_18664\1489119682.py:30: RuntimeWarning: overflow encountered in square
  sse = np.sum(resid**2)
C:\Users\diane\AppData\Local\Temp\ipykernel_18664\1489119682.py:30: RuntimeWarning: overflow encountered in square
  sse = np.sum(resid**2)
C:\Users\diane\AppDa

Saved: Resultats/resultats_all_tickers_ARIMA(2,1,2)_f5years.csv

All ARIMA calculations for 5 YEARS completed successfully!


## SARIMA($p$, $d$, $q$)($P$, $D$, $Q$, $S$)

In [7]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.optimize import minimize
from scipy.signal import lfilter
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
import warnings
import os


class SimpleSARMA:
    def __init__(self, p=1, q=1, P=0, Q=0, s=1):
        self.p = int(p)
        self.q = int(q)
        self.P = int(P)
        self.Q = int(Q)
        self.s = int(s)
        
        self.mu = 0.0
        self.phis = np.zeros(self.p)
        self.thetas = np.zeros(self.q)
        self.Phis = np.zeros(self.P)
        self.Thetas = np.zeros(self.Q)
        
        self.max_ar_lag = self.p + (self.P * self.s)
        self.max_ma_lag = self.q + (self.Q * self.s)
        
        self.last_y_centered = np.zeros(max(1, self.max_ar_lag))
        self.last_resids = np.zeros(max(1, self.max_ma_lag))

    def _build_polynomials(self, phis, thetas, Phis, Thetas):
        ar_poly = np.insert(-phis, 0, 1.0) if self.p > 0 else np.array([1.0])
        
        sar_poly = np.zeros(self.P * self.s + 1)
        sar_poly[0] = 1.0
        for i in range(1, self.P + 1):
            sar_poly[i * self.s] = -Phis[i-1]
            
        full_ar = np.polymul(ar_poly, sar_poly)
        
        ma_poly = np.insert(thetas, 0, 1.0) if self.q > 0 else np.array([1.0])
        
        sma_poly = np.zeros(self.Q * self.s + 1)
        sma_poly[0] = 1.0
        for i in range(1, self.Q + 1):
            sma_poly[i * self.s] = Thetas[i-1]
            
        full_ma = np.polymul(ma_poly, sma_poly)
        return full_ar, full_ma

    def _objective(self, params, y):
        mu = params[0]
        
        idx = 1
        phis = params[idx : idx + self.p]; idx += self.p
        thetas = params[idx : idx + self.q]; idx += self.q
        Phis = params[idx : idx + self.P]; idx += self.P
        Thetas = params[idx : idx + self.Q]
        
        full_ar, full_ma = self._build_polynomials(phis, thetas, Phis, Thetas)
        
        resid = lfilter(full_ma, full_ar, y - mu)
        sse = np.sum(resid**2)
        
        if not np.isfinite(sse) or sse > 1e10:
            return 1e10
        return sse

    def fit(self, y):
        n_params = 1 + self.p + self.q + self.P + self.Q
        initial_guess = np.zeros(n_params)
        initial_guess[0] = np.mean(y)
        
        bounds = [(None, None)] + [(-0.99, 0.99)] * (n_params - 1)
        
        res = minimize(
            self._objective, 
            initial_guess, 
            args=(y,), 
            method='L-BFGS-B',
            bounds=bounds,
            options={'maxiter': 50, 'ftol': 1e-4}
        )
        
        self.mu = res.x[0]
        idx = 1
        self.phis = res.x[idx : idx + self.p]; idx += self.p
        self.thetas = res.x[idx : idx + self.q]; idx += self.q
        self.Phis = res.x[idx : idx + self.P]; idx += self.P
        self.Thetas = res.x[idx : idx + self.Q]
        
        full_ar, full_ma = self._build_polynomials(self.phis, self.thetas, self.Phis, self.Thetas)
        y_centered = y - self.mu
        final_resids = lfilter(full_ma, full_ar, y_centered)
        
        if self.max_ar_lag > 0:
            self.last_y_centered = np.pad(y_centered, (max(0, self.max_ar_lag - len(y_centered)), 0))[-self.max_ar_lag:]
        if self.max_ma_lag > 0:
            self.last_resids = np.pad(final_resids, (max(0, self.max_ma_lag - len(final_resids)), 0))[-self.max_ma_lag:]
            
        return self

    def predict(self, steps=1):
        preds = np.zeros(steps)
        y_hist = np.concatenate([self.last_y_centered, np.zeros(steps)])
        e_hist = np.concatenate([self.last_resids, np.zeros(steps)])
        
        full_ar, full_ma = self._build_polynomials(self.phis, self.thetas, self.Phis, self.Thetas)
        
        ar_coeffs = -full_ar[1:] 
        ma_coeffs = full_ma[1:]
        
        for h in range(steps):
            ar_sum = 0
            if len(ar_coeffs) > 0:
                for i, coef in enumerate(ar_coeffs):
                    ar_sum += coef * y_hist[self.max_ar_lag + h - (i + 1)]
                    
            ma_sum = 0
            if len(ma_coeffs) > 0:
                for j, coef in enumerate(ma_coeffs):
                    ma_sum += coef * e_hist[self.max_ma_lag + h - (j + 1)]
            
            pred_centered = ar_sum + ma_sum
            y_hist[self.max_ar_lag + h] = pred_centered
            preds[h] = self.mu + pred_centered
            
        return preds


class SimpleSARIMA:
    def __init__(self, p=1, d=0, q=1, P=0, D=0, Q=0, s=1):
        self.d, self.D, self.s = int(d), int(D), int(s)
        self.sarma_model = SimpleSARMA(p=p, q=q, P=P, Q=Q, s=s)
        self.history = np.array([]) 

    def fit(self, y):
        required_len = self.d + (self.D * self.s)
        if len(y) <= required_len:
            raise ValueError(f"Pas assez de données pour différencier. Requis: {required_len}, Reçu: {len(y)}")
            
        self.history = y.copy()
        y_diff = y.copy()
        
        for _ in range(self.D):
            y_diff = y_diff[self.s:] - y_diff[:-self.s]
            
        for _ in range(self.d):
            y_diff = np.diff(y_diff)
            
        self.sarma_model.fit(y_diff)
        return self

    def predict(self, steps=1):
        diff_preds = self.sarma_model.predict(steps=steps)
        
        if self.d == 0 and self.D == 0:
            return diff_preds
            
        total_len = len(self.history) + steps
        reconstructed = np.zeros(total_len)
        reconstructed[:len(self.history)] = self.history
        
        current_diff = diff_preds.copy()
        
        if self.d > 0:
            base_series_for_d = self.history.copy()
            for _ in range(self.D):
                base_series_for_d = base_series_for_d[self.s:] - base_series_for_d[:-self.s]
                
            for _ in range(self.d):
                last_val = base_series_for_d[-1]
                current_diff = last_val + np.cumsum(current_diff)
                
        for step in range(steps):
            idx = len(self.history) + step
            val = current_diff[step]
            
            for _ in range(self.D):
                val += reconstructed[idx - self.s]
                
            reconstructed[idx] = val
            
        return reconstructed[len(self.history):]


def wfcv_sarima(y, order, s_order, step_size=50, fold_size=200):
    p, d, q = order
    P, D, Q, s = s_order
    
    y_vals = y.values
    n = len(y_vals)
    preds, truths, mse_tab = [], [], []
    
    model = SimpleSARIMA(p=p, d=d, q=q, P=P, D=D, Q=Q, s=s)
    
    for start in range(0, n - fold_size - step_size + 1, step_size):
        y_train = y_vals[start : start + fold_size]
        y_test = y_vals[start + fold_size : start + fold_size + step_size]
        
        try:
            model.fit(y_train)
            fcst = model.predict(steps=len(y_test))
            mse_tab.append(mean_squared_error(y_test, fcst))
        except Exception:
            fcst = [np.nan] * len(y_test)
            mse_tab.append(np.nan)
            
        preds.extend(fcst)
        truths.extend(y_test)
        
    return np.array(preds), np.array(truths), np.array(mse_tab)

def stats_forecasting_sarima(name_ticker, df_all, order, seasonal_base, s_val, step_size=50, fold_size=200):
    P, D, Q = seasonal_base
    s_order = (P, D, Q, s_val)
    p, d, q = order
    
    df_t = df_all[['Date', name_ticker]].rename(columns={name_ticker: 'return'})
    df_t = df_t[df_t['return'] > -1].copy()
    df_t['log_return'] = np.log1p(df_t['return']).replace([np.inf, -np.inf], np.nan)
    df_t = df_t.dropna(subset=['log_return'])
    y = df_t['log_return']
    
    y_pred, y_true, mse_tab = wfcv_sarima(y, order, s_order, step_size, fold_size)

    mask = np.isfinite(y_pred) & np.isfinite(y_true)
    if mask.sum() >= 3:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            reg = sm.OLS(y_true[mask], sm.add_constant(y_pred[mask])).fit()
            ols_r2 = float(reg.rsquared)
            ols_intercept = float(reg.params[0])
            ols_slope = float(reg.params[1])
            p_int = float(reg.pvalues[0])
            p_slope = float(reg.pvalues[1])
    else:
        ols_r2 = ols_intercept = ols_slope = p_int = p_slope = float('nan')

    return {
        'SYMBOL': name_ticker,
        'Model': f'SARIMA({p},{d},{q})({P},{D},{Q})_{s_val} Custom',
        'p': p, 'd': d, 'q': q,
        'P': P, 'D': D, 'Q': Q, 's': s_val,
        'MSE': float(np.nanmean(mse_tab)) if len(mse_tab) > 0 else np.nan,
        'OLS_R2': ols_r2,
        'OLS_Intercept': ols_intercept,
        'OLS_Slope': ols_slope,
        'OLS_P_Value_Intercept': p_int,
        'P_Value_Slope': p_slope,
        'n_obs': int(len(y)),
    }


### 1 month window

In [8]:
if __name__ == "__main__":
    import os
    from tqdm import tqdm

    FOLD_SIZE = 21
    STEP_SIZE = 5
    # Note: SARIMA needs more buffer due to seasonal lags (s=21). 
    MIN_REQUIS = FOLD_SIZE + STEP_SIZE + 50

    df_all = pd.read_csv('Datasets/returns_all.csv')
    df_all['Date'] = pd.to_datetime(df_all['Date'], errors='coerce')
    all_tickers = df_all.columns[1:].tolist()

    valid_tickers = [col for col in all_tickers if df_all[col].count() >= MIN_REQUIS]
    print(f"Filtering completed: {len(valid_tickers)} tickers kept (those with >= {MIN_REQUIS} valid days).")

    sarima_configs = [
        ((1, 0, 1), (1, 0, 1)),
        ((1, 1, 1), (0, 1, 1))
    ]
    s_values = [5, 21] 
    
    os.makedirs('Resultats', exist_ok=True)

    for order, seasonal_base in sarima_configs:
        for s_val in s_values:
            
            p, d, q = order
            P, D, Q = seasonal_base
            model_name = f"SARIMA({p},{d},{q})({P},{D},{Q})_{s_val}"
            
            output_file = f"Resultats/resultats_{model_name}_f1month.csv"
            rows_for_config = []
            
            for name in tqdm(valid_tickers, desc=f'Training {model_name} — 1 Month', leave=False):
                try:
                    res = stats_forecasting_sarima(
                        name_ticker=name, 
                        df_all=df_all, 
                        order=order, 
                        seasonal_base=seasonal_base,
                        s_val=s_val,
                        step_size=STEP_SIZE, 
                        fold_size=FOLD_SIZE
                    )
                    rows_for_config.append(res)
                except ValueError as e:
                    continue
                except Exception as e:
                    print(f"\n[ERROR] {name} ({model_name}): {e}")
                    continue
                
            df_config = pd.DataFrame(rows_for_config)
            df_config.to_csv(output_file, index=False)
            
            tqdm.write(f"Saved: {output_file}")
            
    print("\nAll SARIMA calculations for 1 MONTH completed successfully!")

Filtering completed: 1063 tickers kept (those with >= 76 valid days).


Saved: Resultats/resultats_SARIMA(1,0,1)(1,0,1)_5_f1month.csv


Saved: Resultats/resultats_SARIMA(1,0,1)(1,0,1)_21_f1month.csv


Saved: Resultats/resultats_SARIMA(1,1,1)(0,1,1)_5_f1month.csv


Training SARIMA(1,1,1)(0,1,1)_21 — 1 Month:   0%|          | 0/1063 [00:00<?, ?it/s]C:\Users\diane\AppData\Local\Temp\ipykernel_18664\803023801.py:246: RuntimeWarning: Mean of empty slice
  'MSE': float(np.nanmean(mse_tab)) if len(mse_tab) > 0 else np.nan,
C:\Users\diane\AppData\Local\Temp\ipykernel_18664\803023801.py:246: RuntimeWarning: Mean of empty slice
  'MSE': float(np.nanmean(mse_tab)) if len(mse_tab) > 0 else np.nan,
C:\Users\diane\AppData\Local\Temp\ipykernel_18664\803023801.py:246: RuntimeWarning: Mean of empty slice
  'MSE': float(np.nanmean(mse_tab)) if len(mse_tab) > 0 else np.nan,
C:\Users\diane\AppData\Local\Temp\ipykernel_18664\803023801.py:246: RuntimeWarning: Mean of empty slice
  'MSE': float(np.nanmean(mse_tab)) if len(mse_tab) > 0 else np.nan,
C:\Users\diane\AppData\Local\Temp\ipykernel_18664\803023801.py:246: RuntimeWarning: Mean of empty slice
  'MSE': float(np.nanmean(mse_tab)) if len(mse_tab) > 0 else np.nan,
C:\Users\diane\AppData\Local\Temp\ipykernel_18664\8

Saved: Resultats/resultats_SARIMA(1,1,1)(0,1,1)_21_f1month.csv

All SARIMA calculations for 1 MONTH completed successfully!


### 1 year window

In [ ]:
if __name__ == "__main__":
    import os
    from tqdm import tqdm

    FOLD_SIZE = 200
    STEP_SIZE = 50
    MIN_REQUIS = FOLD_SIZE + STEP_SIZE + 50 

    df_all = pd.read_csv('Datasets/returns_all.csv')
    df_all['Date'] = pd.to_datetime(df_all['Date'], errors='coerce')
    all_tickers = df_all.columns[1:].tolist()

    valid_tickers = [col for col in all_tickers if df_all[col].count() >= MIN_REQUIS]
    print(f"Filtering completed: {len(valid_tickers)} tickers kept (those with >= {MIN_REQUIS} valid days).")

    sarima_configs = [
        ((1, 0, 1), (1, 0, 1)),
        ((1, 1, 1), (0, 1, 1))
    ]
    s_values = [5, 21] 
    
    os.makedirs('Resultats', exist_ok=True)

    for order, seasonal_base in sarima_configs:
        for s_val in s_values:
            
            p, d, q = order
            P, D, Q = seasonal_base
            model_name = f"SARIMA({p},{d},{q})({P},{D},{Q})_{s_val}"
            
            output_file = f"Resultats/resultats_{model_name}_f1year.csv"
            rows_for_config = []
            
            for name in tqdm(valid_tickers, desc=f'Training {model_name} — 1 Year', leave=False):
                try:
                    res = stats_forecasting_sarima(
                        name_ticker=name, 
                        df_all=df_all, 
                        order=order, 
                        seasonal_base=seasonal_base,
                        s_val=s_val,
                        step_size=STEP_SIZE, 
                        fold_size=FOLD_SIZE
                    )
                    rows_for_config.append(res)
                except ValueError as e:
                    continue
                except Exception as e:
                    print(f"\n[ERROR] {name} ({model_name}): {e}")
                    continue
                
            df_config = pd.DataFrame(rows_for_config)
            df_config.to_csv(output_file, index=False)
            
            tqdm.write(f"Saved: {output_file}")
            
    print("\nAll SARIMA calculations for 1 YEAR completed successfully!")

### 5 years window

In [9]:
if __name__ == "__main__":
    import os
    from tqdm import tqdm

    FOLD_SIZE = 1260
    STEP_SIZE = 250
    MIN_REQUIS = FOLD_SIZE + STEP_SIZE + 100 

    df_all = pd.read_csv('Datasets/returns_all.csv')
    df_all['Date'] = pd.to_datetime(df_all['Date'], errors='coerce')
    all_tickers = df_all.columns[1:].tolist()

    valid_tickers = [col for col in all_tickers if df_all[col].count() >= MIN_REQUIS]
    print(f"Filtering completed: {len(valid_tickers)} tickers kept (those with >= {MIN_REQUIS} valid days).")

    sarima_configs = [
        ((1, 0, 1), (1, 0, 1)),
        ((1, 1, 1), (0, 1, 1))
    ]
    s_values = [5, 21] 
    
    os.makedirs('Resultats', exist_ok=True)

    for order, seasonal_base in sarima_configs:
        for s_val in s_values:
            
            p, d, q = order
            P, D, Q = seasonal_base
            model_name = f"SARIMA({p},{d},{q})({P},{D},{Q})_{s_val}"
            
            output_file = f"Resultats/resultats_{model_name}_f5years.csv"
            rows_for_config = []
            
            # Progress bar shows the model being trained
            for name in tqdm(valid_tickers, desc=f'Training {model_name} — 5 Years', leave=False):
                try:
                    res = stats_forecasting_sarima(
                        name_ticker=name, 
                        df_all=df_all, 
                        order=order, 
                        seasonal_base=seasonal_base,
                        s_val=s_val,
                        step_size=STEP_SIZE, 
                        fold_size=FOLD_SIZE
                    )
                    rows_for_config.append(res)
                except ValueError as e:
                    continue
                except Exception as e:
                    print(f"\n[ERROR] {name} ({model_name}): {e}")
                    continue
                
            df_config = pd.DataFrame(rows_for_config)
            df_config.to_csv(output_file, index=False)
            
            tqdm.write(f"Saved: {output_file}")
            
    print("\nAll SARIMA calculations for 5 YEARS completed successfully!")

Filtering completed: 1012 tickers kept (those with >= 1610 valid days).


Saved: Resultats/resultats_SARIMA(1,0,1)(1,0,1)_5_f5years.csv


Saved: Resultats/resultats_SARIMA(1,0,1)(1,0,1)_21_f5years.csv


Saved: Resultats/resultats_SARIMA(1,1,1)(0,1,1)_5_f5years.csv


Saved: Resultats/resultats_SARIMA(1,1,1)(0,1,1)_21_f5years.csv

All SARIMA calculations for 5 YEARS completed successfully!
